# ☕ 坚果云数据源配置指南

本 Notebook 将引导你完成坚果云数据源的配置，包括：
1. 配置坚果云凭证
2. 测试连接
3. 加载 Excel 数据
4. 预览数据结构

---

## 📋 步骤 1: 准备工作

### 1.1 获取坚果云应用密码

在开始之前，你需要从坚果云获取应用密码：

1. 访问 [坚果云官网](https://www.jianguoyun.com/)
2. 登录你的账号
3. 点击右上角头像 → **账户信息**
4. 选择 **安全选项** 标签
5. 找到 **第三方应用管理** 部分
6. 点击 **添加应用密码**
7. 输入应用名称（例如："Streamlit Dashboard"）
8. 点击 **生成密码**
9. **复制生成的密码**（只显示一次！）

⚠️ **重要提示**：
- 应用密码 ≠ 登录密码
- 应用密码用于第三方应用通过 WebDAV 访问坚果云
- 请妥善保存，密码只显示一次

### 1.2 检查依赖包

In [3]:
# 检查并安装必要的依赖包
import sys

required_packages = {
    'pandas': 'pandas',
    'openpyxl': 'openpyxl',
    'webdav3': 'webdavclient3'
}

missing_packages = []

print("🔍 检查依赖包...\n")

for package, install_name in required_packages.items():
    try:
        __import__(package)
        print(f"✅ {package}")
    except ImportError:
        print(f"❌ {package} - 未安装")
        missing_packages.append(install_name)

if missing_packages:
    print(f"\n⚠️  需要安装以下包: {', '.join(missing_packages)}")
    print(f"\n运行以下命令安装：")
    print(f"!pip install {' '.join(missing_packages)}")
else:
    print("\n🎉 所有依赖包已安装！")

🔍 检查依赖包...

✅ pandas
✅ openpyxl
✅ webdav3

🎉 所有依赖包已安装！


In [2]:
!pip install openpyxl webdavclient3

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached webdavclient3-3.14.6-py3-none-any.whl
  Using cached webdavclient3-3.14.6-py3-none-any.whl
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_universal2.whl.metadata (3.6 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_universal2.whl (8.7 MB)
  Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_universal2.whl.metadata (3.6 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_universal2.whl (8.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openpyxl]3/4 [openpyxl]
   ━━━━━━

---

## 🔐 步骤 2: 配置坚果云凭证

### 方法 1: 直接在代码中配置（仅用于测试）

⚠️ **注意**：这种方法仅用于测试，不要将包含密码的 Notebook 提交到 Git！

### 方法 2: 保存到配置文件（推荐用于生产环境）

In [14]:
# 将凭证保存到 .streamlit/secrets.toml
from pathlib import Path

def save_credentials(email, app_password):
    """保存凭证到配置文件"""
    
    # 获取项目根目录
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    secrets_file = project_root / ".streamlit" / "secrets.toml"
    
    # 确保目录存在
    secrets_file.parent.mkdir(parents=True, exist_ok=True)
    
    # 生成配置内容
    config_content = f"""# 坚果云凭证配置
# ⚠️ 重要：此文件包含敏感信息，请勿提交到 Git 仓库！

[nutstore]
email = "{email}"
app_password = "{app_password}"
hostname = "https://dav.jianguoyun.com/dav/"
"""
    
    # 写入文件
    with open(secrets_file, 'w', encoding='utf-8') as f:
        f.write(config_content)
    
    print(f"✅ 配置已保存到: {secrets_file}")
    print("\n⚠️  此文件已添加到 .gitignore，不会被提交到 Git")
    return secrets_file

# 使用示例（取消注释并填入你的信息）
# secrets_file = save_credentials(
#     email="your-email@example.com",
#     app_password="your-app-password"
# )

print("💡 取消注释上面的代码并填入你的凭证，然后运行此单元格")

💡 取消注释上面的代码并填入你的凭证，然后运行此单元格


---

## 🧪 步骤 3: 测试坚果云连接

In [15]:
# 导入必要的库
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.visualization.nutstore_loader import NutStoreLoader

print("✅ 导入成功！")

✅ 导入成功！


In [16]:
# 创建坚果云加载器
print("🔌 正在连接坚果云...\n")

try:
    loader = NutStoreLoader(
        email=NUTSTORE_EMAIL,
        app_password=NUTSTORE_APP_PASSWORD
    )
    
    # 测试连接
    if loader.check_connection():
        print("✅ 连接成功！")
    else:
        print("❌ 连接失败")
        
except Exception as e:
    print(f"❌ 错误: {str(e)}")

INFO:src.visualization.nutstore_loader:✅ 成功连接到坚果云


🔌 正在连接坚果云...

✅ 连接成功！


### 3.1 列出坚果云文件（可选）

In [17]:
# 列出根目录的文件和文件夹
print("📁 坚果云根目录内容：\n")

try:
    files = loader.list_files()
    
    if files:
        print(f"找到 {len(files)} 个文件/文件夹：\n")
        for i, file in enumerate(files[:20], 1):  # 只显示前20个
            print(f"{i:2d}. {file}")
        
        if len(files) > 20:
            print(f"\n... 还有 {len(files) - 20} 个文件")
    else:
        print("未找到文件")
        
except Exception as e:
    print(f"❌ 错误: {str(e)}")

📁 坚果云根目录内容：

找到 11 个文件/文件夹：

 1. dav/
 2. US Accounting/
 3. Documents/
 4. US Tax/
 5. 投资团队文件/
 6. Maku财务档案/
 7. Tricon Cooperation/
 8. SS 财务档案/
 9. Tristin & Amber/
10. Gondwana/
11. 我的坚果云/


---

## 📊 步骤 4: 加载咖啡进出口数据

In [18]:
# 配置数据文件路径
REMOTE_PATH = "Gondwana/04_Coffee Business 咖啡业务/03 行情报告/10 Import and Price Track/Supply_Demand BS.xlsx"
SHEET_NAMES = ['Demand_Factsheet', 'China_Import', 'China_Export']

print(f"📄 文件路径: {REMOTE_PATH}")
print(f"📋 目标 Sheet: {', '.join(SHEET_NAMES)}")

📄 文件路径: Gondwana/04_Coffee Business 咖啡业务/03 行情报告/10 Import and Price Track/Supply_Demand BS.xlsx
📋 目标 Sheet: Demand_Factsheet, China_Import, China_Export


In [19]:
# 加载 Excel 数据
import pandas as pd

print("📥 正在从坚果云加载数据...\n")

try:
    data_sheets = loader.load_excel(
        remote_path=REMOTE_PATH,
        sheet_names=SHEET_NAMES
    )
    
    print("\n" + "="*70)
    print("✅ 数据加载成功！")
    print("="*70)
    
except Exception as e:
    print(f"❌ 加载失败: {str(e)}")
    import traceback
    traceback.print_exc()

INFO:src.visualization.nutstore_loader:📥 正在下载: Gondwana/04_Coffee Business 咖啡业务/03 行情报告/10 Import and Price Track/Supply_Demand BS.xlsx


📥 正在从坚果云加载数据...



INFO:src.visualization.nutstore_loader:✅ 文件已保存到: data/raw/Supply_Demand BS.xlsx
INFO:src.visualization.nutstore_loader:📖 正在读取 Excel 文件...
INFO:src.visualization.nutstore_loader:📖 正在读取 Excel 文件...
INFO:src.visualization.nutstore_loader:  📊 读取 sheet: Demand_Factsheet
INFO:src.visualization.nutstore_loader:  📊 读取 sheet: Demand_Factsheet
INFO:src.visualization.nutstore_loader:    ✅ 61 行 × 27 列
INFO:src.visualization.nutstore_loader:  📊 读取 sheet: China_Import
INFO:src.visualization.nutstore_loader:    ✅ 61 行 × 27 列
INFO:src.visualization.nutstore_loader:  📊 读取 sheet: China_Import
INFO:src.visualization.nutstore_loader:    ✅ 1410 行 × 12 列
INFO:src.visualization.nutstore_loader:  📊 读取 sheet: China_Export
INFO:src.visualization.nutstore_loader:    ✅ 1410 行 × 12 列
INFO:src.visualization.nutstore_loader:  📊 读取 sheet: China_Export
INFO:src.visualization.nutstore_loader:    ✅ 1164 行 × 8 列
INFO:src.visualization.nutstore_loader:✅ 成功加载 3 个数据表
INFO:src.visualization.nutstore_loader:    ✅ 1164 行 × 8 列


✅ 数据加载成功！


---

## 📈 步骤 5: 数据预览和分析

### 5.1 数据概览

In [20]:
# 显示所有 Sheet 的基本信息
print("="*70)
print("📊 数据概览")
print("="*70 + "\n")

for sheet_name, df in data_sheets.items():
    print(f"📋 Sheet: {sheet_name}")
    print(f"   维度: {len(df)} 行 × {len(df.columns)} 列")
    print(f"   内存: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    print()

📊 数据概览

📋 Sheet: Demand_Factsheet
   维度: 61 行 × 27 列
   内存: 13.00 KB

📋 Sheet: China_Import
   维度: 1410 行 × 12 列
   内存: 300.49 KB

📋 Sheet: China_Export
   维度: 1164 行 × 8 列
   内存: 160.24 KB



### 5.2 Demand_Factsheet (宽表)

In [21]:
# 查看 Demand_Factsheet
df_demand = data_sheets['Demand_Factsheet']

print("📊 Demand_Factsheet 数据结构")
print("="*70 + "\n")

print(f"列名 ({len(df_demand.columns)} 列):")
for i, col in enumerate(df_demand.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n数据类型:")
print(df_demand.dtypes)

print(f"\n前 5 行数据:")
df_demand.head()

📊 Demand_Factsheet 数据结构

列名 (27 列):
   1. Date
   2. Brazil
   3. Colombia
   4. Viet Nam
   5. Uganda
   6. Ethiopia
   7. Middle America
   8. Indonesia
   9. Other
  10. Total Import
  11. Yunnan Demand
  12. Yunnan Export
  13. Yunnan Production
  14. Viet Nam Export
  15. Total Real Demand
  16. Demand Percent
  17. Brazil.1
  18. Colombia.1
  19. Viet Nam.1
  20. Uganda.1
  21. Ethiopia.1
  22. Middle America.1
  23. Indonesia.1
  24. Other.1
  25. Yunnan
  26. Unnamed: 25
  27. Lastest

数据类型:
Date                 datetime64[ns]
Brazil                      float64
Colombia                    float64
Viet Nam                    float64
Uganda                      float64
Ethiopia                    float64
Middle America              float64
Indonesia                   float64
Other                       float64
Total Import                float64
Yunnan Demand               float64
Yunnan Export               float64
Yunnan Production           float64
Viet Nam Export            

,Date,Brazil,Colombia,Viet Nam,Uganda,Ethiopia,Middle America,Indonesia,Other,Total Import,...,Colombia.1,Viet Nam.1,Uganda.1,Ethiopia.1,Middle America.1,Indonesia.1,Other.1,Yunnan,Unnamed: 25,Lastest
0,2021-01-01,1672410.0,1244694.0,3615786.0,1213212.0,654200.0,47923.0,927995.0,632922.0,10009142.0,...,0.056026,0.272860,0.054609,0.029447,0.002157,0.041771,0.028489,0.439365,NaN,2025-10-01
1,2021-02-01,1193097.0,688762.0,5367004.0,465610.0,517128.0,87971.0,166504.0,279498.0,8765574.0,...,0.034142,0.356904,0.023080,0.025634,0.004361,0.008254,0.013855,0.474629,NaN,NaT
2,2021-03-01,1705422.0,1118735.0,3044312.0,166510.0,612173.0,1482430.0,277777.0,244529.0,8651888.0,...,0.068239,0.147124,0.010157,0.037341,0.090423,0.016943,0.014915,0.510833,NaN,NaT
3,2021-04-01,1667682.0,1116984.0,2635921.0,421360.0,616663.0,1302298.0,156581.0,295249.0,8212738.0,...,0.062518,0.319254,0.023584,0.034515,0.072890,0.008764,0.016525,0.368610,NaN,NaT
4,2021-05-01,1910524.0,762975.0,2584694.0,155293.0,1293788.0,1465547.0,246144.0,174064.0,8593029.0,...,0.043147,0.253178,0.008782,0.073165,0.082878,0.013920,0.009843,0.407045,NaN,NaT


In [22]:
# 数据统计
print("📈 Demand_Factsheet 数值列统计\n")
df_demand.describe()

📈 Demand_Factsheet 数值列统计



,Date,Brazil,Colombia,Viet Nam,Uganda,Ethiopia,Middle America,Indonesia,Other,Total Import,...,Colombia.1,Viet Nam.1,Uganda.1,Ethiopia.1,Middle America.1,Indonesia.1,Other.1,Yunnan,Unnamed: 25,Lastest
count,61,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01,...,60.000000,60.000000,6.000000e+01,60.000000,60.000000,60.000000,60.000000,60.000000,0.0,1
mean,2023-07-02 02:21:38.360655616,3.909507e+06,1.710538e+06,1.935300e+06,5.332942e+05,2.700518e+06,6.148349e+05,7.160452e+05,5.540047e+05,1.267404e+07,...,0.089830,0.241209,2.934637e-02,0.146623,0.041954,0.041824,0.031547,0.180003,NaN,2025-10-01 00:00:00
min,2021-01-01 00:00:00,2.766140e+05,1.413410e+05,1.937700e+04,5.000000e+00,5.171280e+05,4.700000e+01,1.044760e+05,9.168800e+04,2.059702e+06,...,0.013987,0.060748,3.607347e-07,0.019742,0.000003,0.005232,0.006458,0.000000,NaN,2025-10-01 00:00:00
25%,2022-04-01 00:00:00,1.776645e+06,8.790815e+05,1.229129e+06,2.262055e+05,9.108070e+05,1.470455e+05,2.289538e+05,3.057332e+05,8.637173e+06,...,0.052650,0.165451,1.259728e-02,0.062147,0.007330,0.015510,0.016929,0.000000,NaN,2025-10-01 00:00:00
50%,2023-07-01 00:00:00,2.709404e+06,1.228420e+06,1.933956e+06,3.908780e+05,1.741251e+06,2.981245e+05,4.629410e+05,4.528895e+05,1.071154e+07,...,0.079989,0.228973,2.339329e-02,0.109238,0.020608,0.024147,0.024805,0.087653,NaN,2025-10-01 00:00:00
75%,2024-10-01 00:00:00,5.101763e+06,1.869698e+06,2.530630e+06,7.038060e+05,2.944855e+06,8.416522e+05,1.180744e+06,6.615660e+05,1.627280e+07,...,0.116777,0.307863,3.883663e-02,0.179216,0.043331,0.048597,0.035817,0.352809,NaN,2025-10-01 00:00:00
max,2026-01-01 00:00:00,2.352263e+07,7.922058e+06,5.367004e+06,1.924106e+06,1.351236e+07,2.527507e+06,2.925993e+06,3.725779e+06,3.714333e+07,...,0.246369,0.498483,9.040575e-02,0.484123,0.214530,0.206082,0.182180,0.732044,NaN,2025-10-01 00:00:00
std,NaN,3.621434e+06,1.478173e+06,1.034371e+06,4.625003e+05,2.756736e+06,6.640913e+05,6.582601e+05,5.021076e+05,6.010661e+06,...,0.051985,0.096470,2.280596e-02,0.118192,0.054460,0.043009,0.026025,0.205457,NaN,NaN


### 5.3 China_Import (长表)

In [23]:
# 查看 China_Import
df_import = data_sheets['China_Import']

print("📊 China_Import 数据结构")
print("="*70 + "\n")

print(f"列名 ({len(df_import.columns)} 列):")
for i, col in enumerate(df_import.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n数据类型:")
print(df_import.dtypes)

print(f"\n前 10 行数据:")
df_import.head(10)

📊 China_Import 数据结构

列名 (12 列):
   1. Date
   2. Date_Org
   3. Country
   4. Import (kg)
   5. Value (USD)
   6. Unnamed: 5
   7. Unnamed: 6
   8. Row Labels
   9. Sum of Import (kg)
  10. Unnamed: 9
  11. Middle America Coffee Export Countries Fisrt 3 Rank:
  12. Unnamed: 11

数据类型:
Date                                                    datetime64[ns]
Date_Org                                                         int64
Country                                                         object
Import (kg)                                                      int64
Value (USD)                                                    float64
Unnamed: 5                                                     float64
Unnamed: 6                                                     float64
Row Labels                                                      object
Sum of Import (kg)                                             float64
Unnamed: 9                                                     float64
Middl

,Date,Date_Org,Country,Import (kg),Value (USD),Unnamed: 5,Unnamed: 6,Row Labels,Sum of Import (kg),Unnamed: 9,Middle America Coffee Export Countries Fisrt 3 Rank:,Unnamed: 11
0,2021-01-01,202101,India,19800,53809.0,NaN,NaN,<2021/1/1,NaN,NaN,Honduras,Honduras
1,2021-01-01,202101,Indonesia,927995,2585583.0,NaN,NaN,<2021/1/1,NaN,NaN,Guatemala,Guatemala
2,2021-01-01,202101,Malaysia,1,31.0,NaN,NaN,2021,106472492.0,NaN,Costa Rica,Costa Rica
3,2021-01-01,202101,Viet Nam,3615786,5599477.0,NaN,NaN,Jan,10009142.0,NaN,NaN,NaN
4,2021-01-01,202101,China,3,72.0,NaN,NaN,Feb,8765574.0,NaN,NaN,NaN
5,2021-01-01,202101,"Taiwan,China",3255,7631.0,NaN,NaN,Mar,8651888.0,NaN,NaN,NaN
6,2021-01-01,202101,Timor-Leste,48900,290435.0,NaN,NaN,Apr,8212738.0,NaN,NaN,NaN
7,2021-01-01,202101,Cameroon,34,28.0,NaN,NaN,May,8593029.0,NaN,NaN,NaN
8,2021-01-01,202101,Ethiopia,654200,2427288.0,NaN,NaN,Jun,10230714.0,NaN,NaN,NaN
9,2021-01-01,202101,Kenya,20413,129331.0,NaN,NaN,Jul,8071187.0,NaN,NaN,NaN


In [24]:
# 检查唯一值
print("🔍 China_Import 数据探索\n")

for col in df_import.columns:
    unique_count = df_import[col].nunique()
    null_count = df_import[col].isnull().sum()
    
    print(f"{col}:")
    print(f"  唯一值: {unique_count}")
    print(f"  缺失值: {null_count}")
    
    if unique_count < 20:  # 如果唯一值少于20个，显示所有值
        print(f"  值列表: {df_import[col].unique().tolist()}")
    print()

🔍 China_Import 数据探索

Date:
  唯一值: 57
  缺失值: 0

Date_Org:
  唯一值: 57
  缺失值: 0

Country:
  唯一值: 57
  缺失值: 0

Import (kg):
  唯一值: 1116
  缺失值: 0

Value (USD):
  唯一值: 1306
  缺失值: 43

Unnamed: 5:
  唯一值: 0
  缺失值: 1410
  值列表: [nan]

Unnamed: 6:
  唯一值: 0
  缺失值: 1410
  值列表: [nan]

Row Labels:
  唯一值: 19
  缺失值: 1345
  值列表: ['<2021/1/1', '2021', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', '2022', '2023', '2024', '2025', 'Grand Total', nan]

Sum of Import (kg):
  唯一值: 63
  缺失值: 1347

Unnamed: 9:
  唯一值: 0
  缺失值: 1410
  值列表: [nan]

Middle America Coffee Export Countries Fisrt 3 Rank::
  唯一值: 3
  缺失值: 1407
  值列表: ['Honduras', 'Guatemala', 'Costa Rica', nan]

Unnamed: 11:
  唯一值: 3
  缺失值: 1407
  值列表: ['Honduras', 'Guatemala', 'Costa Rica', nan]



### 5.4 China_Export (长表)

In [25]:
# 查看 China_Export
df_export = data_sheets['China_Export']

print("📊 China_Export 数据结构")
print("="*70 + "\n")

print(f"列名 ({len(df_export.columns)} 列):")
for i, col in enumerate(df_export.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n数据类型:")
print(df_export.dtypes)

print(f"\n前 10 行数据:")
df_export.head(10)

📊 China_Export 数据结构

列名 (8 列):
   1. Date
   2. Date_Org
   3. Trading partner
   4. Quantity
   5. US dollar
   6. Unnamed: 5
   7. Row Labels
   8. Sum of Quantity

数据类型:
Date               datetime64[ns]
Date_Org                    int64
Trading partner            object
Quantity                    int64
US dollar                   int64
Unnamed: 5                float64
Row Labels                 object
Sum of Quantity           float64
dtype: object

前 10 行数据:


,Date,Date_Org,Trading partner,Quantity,US dollar,Unnamed: 5,Row Labels,Sum of Quantity
0,2020-01-01,202001,Hong Kong£¨China,14120,41184,NaN,<2020/1/1,NaN
1,2020-01-01,202001,Japan,77550,112718,NaN,2020,49169113.0
2,2020-01-01,202001,Malaysia,38400,112593,NaN,Jan,724405.0
3,2020-01-01,202001,Republic of Korea,540,4428,NaN,Feb,446020.0
4,2020-01-01,202001,Syria,78000,123850,NaN,Mar,2738296.0
5,2020-01-01,202001,Thailand,50040,142025,NaN,Apr,11311625.0
6,2020-01-01,202001,Yemen,20000,34345,NaN,May,9623657.0
7,2020-01-01,202001,Taiwan£¨China,44715,150787,NaN,Jun,7815829.0
8,2020-01-01,202001,Uzbekistan,2040,6468,NaN,Jul,7836355.0
9,2020-01-01,202001,Germany,129600,312802,NaN,Aug,3566990.0


---

## 📊 步骤 6: 快速可视化测试

In [26]:
# 安装可视化库（如果需要）
try:
    import plotly.express as px
    print("✅ Plotly 已安装")
except ImportError:
    print("📦 安装 Plotly...")
    !pip install plotly
    import plotly.express as px

📦 安装 Plotly...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/9.8 MB ? eta -:--:--Downloading plotly-6.3.1-py3-none-any.whl (9.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 4.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 4.7 MB/s  0:00:02m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]


In [27]:
# 简单可视化示例
# 注意：这里需要根据实际的列名进行调整

import plotly.express as px

print("📊 创建示例图表...\n")

# 检查数据框是否有日期列和数值列
print("提示: 根据实际列名调整以下代码\n")
print("Import 数据列名:", df_import.columns.tolist())
print("Export 数据列名:", df_export.columns.tolist())

📊 创建示例图表...

提示: 根据实际列名调整以下代码

Import 数据列名: ['Date', 'Date_Org', 'Country', 'Import (kg)', 'Value (USD)', 'Unnamed: 5', 'Unnamed: 6', 'Row Labels', 'Sum of Import (kg)', 'Unnamed: 9', 'Middle America Coffee Export Countries Fisrt 3 Rank:', 'Unnamed: 11']
Export 数据列名: ['Date', 'Date_Org', 'Trading partner', 'Quantity', 'US dollar', 'Unnamed: 5', 'Row Labels', 'Sum of Quantity']


---

## ✅ 步骤 7: 配置验证和总结

In [30]:
# 最终验证
print("="*70)
print("🎉 配置验证总结")
print("="*70 + "\n")

checks = [
    ("坚果云连接", loader.check_connection()),
    ("数据加载", len(data_sheets) == 3),
    ("Demand_Factsheet", 'Demand_Factsheet' in data_sheets),
    ("China_Import", 'China_Import' in data_sheets),
    ("China_Export", 'China_Export' in data_sheets),
]

all_passed = True
for check_name, check_result in checks:
    status = "✅" if check_result else "❌"
    print(f"{status} {check_name}")
    all_passed = all_passed and check_result

print("\n" + "="*70)

if all_passed:
    print("🎊 所有检查通过！配置完成！")
    print("\n下一步：")
    print("1. 运行可视化仪表板:")
    print("   streamlit run dashboards/coffee_import_export/app.py")
    print("\n2. 或继续在此 Notebook 中进行数据分析")
else:
    print("⚠️  部分检查未通过，请检查配置")

print("="*70)

🎉 配置验证总结

✅ 坚果云连接
✅ 数据加载
✅ Demand_Factsheet
✅ China_Import
✅ China_Export

🎊 所有检查通过！配置完成！

下一步：
1. 运行可视化仪表板:
   streamlit run dashboards/coffee_import_export/app.py

2. 或继续在此 Notebook 中进行数据分析


---

## 📝 配置信息保存

### 数据文件信息

In [29]:
# 保存数据结构信息
import json
from datetime import datetime

data_info = {
    "last_updated": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "remote_path": REMOTE_PATH,
    "sheets": {}
}

for sheet_name, df in data_sheets.items():
    data_info["sheets"][sheet_name] = {
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": df.columns.tolist(),
        "dtypes": df.dtypes.astype(str).to_dict()
    }

# 保存到文件
output_path = project_root / "data" / "interim" / "data_structure_info.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data_info, f, indent=2, ensure_ascii=False)

print(f"✅ 数据结构信息已保存到: {output_path}")

✅ 数据结构信息已保存到: /Users/caddyzhang/Documents/X_Codes/Caddy's data/data/interim/data_structure_info.json


---

## 🎯 下一步

配置完成后，你可以：

1. **运行可视化仪表板**
   ```bash
   streamlit run dashboards/coffee_import_export/app.py
   ```

2. **在此 Notebook 中继续数据分析**
   - 数据清洗
   - 探索性数据分析
   - 创建自定义图表

3. **查看详细文档**
   - `NEXT_STEPS.md` - 下一步指南
   - `dashboards/coffee_import_export/README.md` - 详细配置说明

---

**恭喜！坚果云数据源配置完成！** 🎉